# Exercise: Image Classification with CNNs

In this exercise, we want to look at image classification again, but this time use a convolutional neural network for the cifar-10 data set. We will use torch.

We have to install some additional packages that print some more information about the network.

In [ ]:
!pip install torchinfo torcheval

In [ ]:
import cv2
import os
import sys
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
from torcheval.metrics import MulticlassAccuracy
import torchvision


sys.path.append('.')
from cifar import load_cifar

# for displaying images in jupyter
import matplotlib as mpl
from matplotlib import pyplot as plt

from pathlib import Path

data_dir = Path('/exchange/cvai/images')

## Exercise 1: Image Classification on CIFAR-10

The data set that we use is again the CIFAR data set. It contains 60000 small (32x32) images of 10 different classes of which 50000 are in the training set and 10000 in the test set. There is a version of the data set installed in the /exchange/cvai/images folder. We will use that, but take the data loader from torchvision and just tell it that it is already downloaded.


In [ ]:
transform = torchvision.transforms.Compose(
    [torchvision.transforms.ToTensor()])
data_train = torchvision.datasets.CIFAR10(root=data_dir, download=False, transform=transform)
data_test = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=False, transform=transform)

In [ ]:
print (f'train: {len(data_train)}')
print (f'train: {len(data_test)}')

In [ ]:
BATCH_SIZE = 64

data_train_loader = torch.utils.data.DataLoader(dataset=data_train, shuffle=True, batch_size=BATCH_SIZE)
data_test_loader = torch.utils.data.DataLoader(dataset=data_test, shuffle=False, batch_size=BATCH_SIZE)


In [ ]:
train_iter = iter(data_train_loader)
images, labels = next(train_iter)

plt.imshow(np.transpose(torchvision.utils.make_grid(images), (1, 2, 0)))

### Design neural network

We now have to design the neural network in torch. For that, we should implement a class that is derived from `nn.Module`.

We define the network parts usually in the constructor and call them in the forward method. The easiest solution is to use a `nn.Sequential` object, but there are also other possibilities.

In [ ]:
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()
        

    def forward(self, x):

        # YOUR CODE HERE
        raise NotImplementedError()




In [ ]:
my_cnn = MyCNN()
# 
# Print gives some information about the cnn
#
print(my_cnn)
# ... but the output of summary (from torchinfo) gives some extra information
summary(my_cnn, input_size=(64,3, 32,32))

In [ ]:
def get_device():
    if torch.cuda.is_available():
        device = torch.device('cuda')
        # test if it worked
        x = torch.ones(1, device=device)
        print('Using CUDA device')

    elif torch.backends.mps.is_available():
        device = torch.device('mps')
        x = torch.ones(1, device=device)
        print('Using MPS device')
    else:
        print('Using CPU')
        device = torch.device('cpu')
    return device

In [ ]:
device = get_device()

### Training Loop

In torch, we have to define our own training loop.

In [ ]:
def train(epochs: int, model, train_data, val_data, loss_function, optimizer, metrics, device):                                              
    input_count = 0
    step_count = 0
    model = model.to(device)
    
    for epoch in range(epochs):
        model.train()
        metrics.reset()
        for step, (inputs, labels) in enumerate(train_data):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Zero your gradients for every batch!
            optimizer.zero_grad()
            # calculate results
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            train_loss = loss_function(outputs, labels)
            train_loss.backward()
            optimizer.step()

            metrics.update(predicted, labels)
            train_acc = metrics.compute()
           
        model.eval()
        metrics.reset()
        val_loss = []
        val_steps = 0
        for step, (inputs, labels) in enumerate(val_data):
            inputs = inputs.to(device)
            labels = labels.to(device)
            with torch.no_grad():
                outputs = model(inputs)
                _, predicted = torch.max(outputs, 1)

                val_loss.append(loss_function(outputs, labels).item())
                metrics.update(predicted, labels)

        val_acc = metrics.compute()
        val_loss_mean = np.mean(val_loss)

        print(f"Epoch {epoch:02} Train Loss: {train_loss:.3f}, Valid Loss: {val_loss_mean:.3f}, Train Accuracy: {train_acc:.2f} Valid Acc: {val_acc:.2f}")
    # return the last accuracy from the evaluation

                       

In [ ]:
my_cnn = MyCNN()
my_metrics = MulticlassAccuracy(num_classes=10)

# initialize an optimizer and loss function and call train
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# you should be able to get about 60% easily :-)
assert my_metrics.compute() > 0.6